In [3]:
# Cell 1: Imports
import sys
sys.path.append('..')
from src.vectorstore import MovieVectorStore, load_documents_from_json
import json

f:\JSOM MSBUAN\Spring 2026\BUAN 6V99 AI Agentic\Git\projects\agentic-rag-movie-recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Cell 2: Load existing vector store
print("Loading existing vector store...")
vector_store = MovieVectorStore(
    persist_directory="../data/vectorstore",
    collection_name="netflix_gpt_movies"
)
vector_store.collection = vector_store.client.get_collection("netflix_gpt_movies")
print("✅ Vector store loaded")

Loading existing vector store...
Loading embedding model: all-MiniLM-L6-v2
✅ Embedding model loaded
✅ Vector store loaded


In [5]:
# Cell 3: Check collection stats
stats = vector_store.get_collection_stats()
print("Collection Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

Collection Statistics:
  collection_name: netflix_gpt_movies
  total_chunks: 29
  persist_directory: ..\data\vectorstore
  embedding_model: all-MiniLM-L6-v2


In [6]:
# Cell 4: Test basic search
query = "exciting action movies"
results = vector_store.search(query, n_results=5)

print(f"\nQuery: '{query}'")
print(f"Results found: {len(results['documents'][0])}\n")

for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f"{i+1}. Title: {metadata.get('title', 'Unknown')}")
    print(f"   Similarity Score: {1 - distance:.4f}")
    print(f"   Genres: {metadata.get('genres', [])}")
    print(f"   Preview: {doc[:150]}...")
    print()


Query: 'exciting action movies'
Results found: 5

1. Title: Action & Adventure Collection
   Similarity Score: -0.2059
   Genres: []
   Preview: # Action & Adventure Movies Collection

This collection features top-rated action & adventure movies from our catalog. ## Featured Action & Adventure ...

2. Title: Dramas Collection
   Similarity Score: -0.2513
   Genres: []
   Preview: - **Jaws 2** (1978): Four years after the last deadly shark attacks, police chief Martin Brody fights to protect Amity Island from another killer grea...

3. Title: International Movies Collection
   Similarity Score: -0.2537
   Genres: []
   Preview: - **The Father Who Moves Mountains** (2021): When his son goes missing during a snowy hike in the mountains, a retired intelligence officer will stop ...

4. Title: Documentaries Collection
   Similarity Score: -0.2540
   Genres: []
   Preview: ## Genre Characteristics
Documentaries films are known for their distinct storytelling style and audience appeal. ...



In [7]:
# Cell 5: Test with metadata filtering
# Search for Action movies only
query = "best movies"
filter_dict = {"genre": "Action"}  # Note: This depends on your metadata structure

print(f"\nQuery: '{query}' with filter: {filter_dict}")

# If genres are stored as lists, filtering might not work directly
# Let's just do a regular search
results = vector_store.search(query, n_results=5)

for i, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    genres = metadata.get('genres', [])
    if isinstance(genres, list) and 'Action' in genres:
        print(f"{i+1}. {metadata.get('title', 'Unknown')} - {genres}")


Query: 'best movies' with filter: {'genre': 'Action'}


In [8]:
# Cell 6: Test multiple queries
test_queries = [
    "romantic movies for date night",
    "mind-bending psychological thrillers",
    "family friendly comedies",
    "epic science fiction adventures",
    "award winning dramas"
]

print("\n" + "="*70)
print("TESTING MULTIPLE QUERIES")
print("="*70)

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    results = vector_store.search(query, n_results=3)
    
    for i, metadata in enumerate(results['metadatas'][0]):
        title = metadata.get('title', 'Unknown')
        year = metadata.get('year', 'N/A')
        print(f"   {i+1}. {title} ({year})")


TESTING MULTIPLE QUERIES

🔍 Query: 'romantic movies for date night'
   1. The Shawshank Redemption (1994)
   2. Dramas Collection (N/A)
   3. Pulp Fiction (1994)

🔍 Query: 'mind-bending psychological thrillers'
   1. Documentaries Collection (N/A)
   2. International Movies Collection (N/A)
   3. Dramas Collection (N/A)

🔍 Query: 'family friendly comedies'
   1. Comedies Collection (N/A)
   2. Dramas Collection (N/A)
   3. Comedies Collection (N/A)

🔍 Query: 'epic science fiction adventures'
   1. Documentaries Collection (N/A)
   2. Action & Adventure Collection (N/A)
   3. Dramas Collection (N/A)

🔍 Query: 'award winning dramas'
   1. Documentaries Collection (N/A)
   2. Dramas Collection (N/A)
   3. Dramas Collection (N/A)


In [9]:
# Cell 7: Inspect a specific document
# Get all unique titles
results = vector_store.collection.get()
unique_titles = set()
for metadata in results['metadatas']:
    title = metadata.get('title', '')
    if title and 'Collection' not in title:
        unique_titles.add(title)

print(f"\n📚 Total unique movies in database: {len(unique_titles)}")
print("\nSample titles:")
for title in list(unique_titles)[:10]:
    print(f"  - {title}")


📚 Total unique movies in database: 12

Sample titles:
  - The Godfather
  - Spirited Away
  - The Godfather: Part II
  - Schindler's List
  - The Shawshank Redemption
  - Fight Club
  - The Dark Knight
  - The Green Mile
  - Forrest Gump
  - Once Upon a Time in America


In [10]:
# Cell 8: Test semantic understanding
# Test if the vector store understands concepts
semantic_tests = [
    ("movies about artificial intelligence", "Should find sci-fi/AI movies"),
    ("feel-good uplifting films", "Should find comedies/dramas"),
    ("dark and gritty crime movies", "Should find crime/thriller"),
]

print("\n" + "="*70)
print("SEMANTIC UNDERSTANDING TEST")
print("="*70)

for query, expectation in semantic_tests:
    print(f"\n🧠 Query: '{query}'")
    print(f"   Expected: {expectation}")
    results = vector_store.search(query, n_results=2)
    print(f"   Results:")
    for metadata in results['metadatas'][0]:
        print(f"     - {metadata.get('title', 'Unknown')} | Genres: {metadata.get('genres', [])}")


SEMANTIC UNDERSTANDING TEST

🧠 Query: 'movies about artificial intelligence'
   Expected: Should find sci-fi/AI movies
   Results:
     - Schindler's List | Genres: ['Drama', 'History', 'War']
     - Comedies Collection | Genres: []

🧠 Query: 'feel-good uplifting films'
   Expected: Should find comedies/dramas
   Results:
     - Documentaries Collection | Genres: []
     - International Movies Collection | Genres: []

🧠 Query: 'dark and gritty crime movies'
   Expected: Should find crime/thriller
   Results:
     - Pulp Fiction | Genres: ['Thriller', 'Crime']
     - The Dark Knight | Genres: ['Drama', 'Action', 'Crime', 'Thriller']


In [11]:
# Cell 9: Performance check
import time

queries = [
    "action adventure",
    "romantic comedy", 
    "sci-fi thriller",
    "drama film",
    "horror movie"
]

print("\n⚡ Performance Test")
times = []

for query in queries:
    start = time.time()
    results = vector_store.search(query, n_results=5)
    end = time.time()
    times.append(end - start)
    print(f"Query '{query}': {(end-start)*1000:.2f}ms")

print(f"\nAverage query time: {sum(times)/len(times)*1000:.2f}ms")


⚡ Performance Test
Query 'action adventure': 13.92ms
Query 'romantic comedy': 21.66ms
Query 'sci-fi thriller': 13.92ms
Query 'drama film': 13.49ms
Query 'horror movie': 17.59ms

Average query time: 16.12ms
